In [1]:
import matplotlib.pyplot as plt
%matplotlib widget
import pandas
import math

def plot(df):
    fig = plt.figure()
    ax = plt.axes(projection='3d')
    ax.scatter3D(df['X'], df['Y'], df['Z'], marker = 1)
    ax.scatter3D(0,0,250,marker=1)
    fig.show()

# Read data
data = []
for i in range(10):
    data.append(pandas.read_csv('data'+str(i+1)+'.csv').drop('Unnamed: 0', axis = 1))

complete_coord_data = pandas.concat(data, ignore_index = True)
plot(complete_coord_data)

Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

In [2]:
def add_points(m,point):
    keys = m.keys()
    for k in keys:
        if k.startswith('DB'):
            m[k]+=10**(point[k]/10)
        else:
            m[k]+=point[k]
    return m

def average_mean(m,count):
    keys = m.keys()
    for k in keys:
        if k.startswith('DB'):
            m[k] = 10*(math.log10(m[k]/count))
        else:
            m[k] = m[k]/count
    return m

def find_points(x, y):
    points = []
    for elevation in data:
        ind = []
        maxe = 10 #name
        for i,point in elevation.iterrows():
            if abs(x-point['X'])<=1 or abs(y-point['Y'])<=1:
                e = (x-point['X'])**2 + (y-point['Y'])**2
                if e<maxe:
                    maxe = e
                    ind = []
                    ind.append(i)
                elif(e==maxe):
                    ind.append(i)
        if len(ind)==1:
            points.append(elevation.iloc[ind[0]].to_dict())
        elif len(ind)>1:
            mean = {'X':0,'Y':0,'Z':0,'DBZ':0,'DBVEL':0,'DBWIDTH':0}
            count = 0
            for i in ind:
                count+=1
                point = d.iloc[i].to_dict()
                mean = add_points(mean,point)
            mean = average_mean(mean,count)
            points.append(mean)
        else:
            points.append(None)
    return points

In [4]:
x = 3
y = 3

xy_points = find_points(x,y)
xy_points

[{'X': 2.828688091498366,
  'Y': 2.828145749344697,
  'Z': 0.0107378526183838,
  'DBZ': -32.0,
  'DBVEL': 0.0,
  'DBWIDTH': 0.0},
 {'X': 2.827193230342683,
  'Y': 2.8290912436013254,
  'Z': 0.0567553846151453,
  'DBZ': -32.0,
  'DBVEL': 0.0,
  'DBWIDTH': 2.50564453125},
 {'X': 2.827333664198924,
  'Y': 2.826791581728541,
  'Z': 0.1242324625697388,
  'DBZ': -32.0,
  'DBVEL': 0.0,
  'DBWIDTH': 1.77228515625},
 {'X': 2.825073175795016,
  'Y': 2.825073175795016,
  'Z': 0.1947385498737557,
  'DBZ': -32.0,
  'DBVEL': 0.0,
  'DBWIDTH': 2.26119140625},
 {'X': 3.1733001846449684,
  'Y': 3.1726917702844206,
  'Z': 0.3379243538156459,
  'DBZ': -10.0,
  'DBVEL': 2.2174015748031497,
  'DBWIDTH': 2.38341796875},
 {'X': 3.166113249521761,
  'Y': 3.165506213106659,
  'Z': 0.4530974574921928,
  'DBZ': -32.0,
  'DBVEL': 0.0,
  'DBWIDTH': 3.3001171875},
 {'X': 3.1449177965174395,
  'Y': 3.144314823889952,
  'Z': 0.6875873336629954,
  'DBZ': -7.5,
  'DBVEL': -0.3695669291338583,
  'DBWIDTH': 1.03892578125

In [5]:
brightest_point = None

for point in xy_points:
    if point is None:
        continue
    if brightest_point is None:
        brightest_point = point
    elif point['DBZ']>brightest_point['DBZ']:
        brightest_point = point

if brightest_point is None:
    print("No Bright point found!")
    quit()

bright_threshold = brightest_point['DBZ']-5;
bright_points = []
        
for point in xy_points:
    if point is None:
        continue
    elif bright_threshold-point['DBZ']<=5:
        bright_points.append(point)

print(brightest_point)
print(bright_points[0])
print(bright_points[-1])
print(len(bright_points))
print('Base of Brightest Echo : ',bright_points[0]['Z'])
print('Height of Brightest Echo : ',bright_points[-1]['Z'])

{'X': 3.1449177965174395, 'Y': 3.144314823889952, 'Z': 0.6875873336629954, 'DBZ': -7.5, 'DBVEL': -0.3695669291338583, 'DBWIDTH': 1.03892578125}
{'X': 3.1733001846449684, 'Y': 3.1726917702844206, 'Z': 0.3379243538156459, 'DBZ': -10.0, 'DBVEL': 2.2174015748031497, 'DBWIDTH': 2.38341796875}
{'X': 3.114942830866866, 'Y': 3.114047035474794, 'Z': 0.9218688742341647, 'DBZ': -14.5, 'DBVEL': -2.2174015748031497, 'DBWIDTH': 1.1000390625}
3
Base of Brightest Echo :  0.3379243538156459
Height of Brightest Echo :  0.9218688742341647


In [6]:
average_echo = 0
number_of_valid_points = 0

for point in xy_points:
    if point is None:
        continue
    if point['DBZ']==-32:
        continue
    average_echo += 10**(point['DBZ']/10)
    number_of_valid_points += 1
average_echo /= number_of_valid_points
average_echo = 10*(math.log10(average_echo))

average_threshold = average_echo-5
average_points = []

for point in xy_points:
    if point['DBZ']>=average_threshold:
        average_points.append(point)

highest_average = average_points[-1]
lowest_average = average_points[0]
echo_depth = highest_average['Z']-lowest_average['Z']
print('Depth of Average Echo : ',echo_depth)

Depth of Average Echo :  0.5839445204185189


In [9]:
mean_vel = 0
mean_width = 0
number_of_points = 0

for point in xy_points:
    if point is None:
        continue
    if point['Z']>2.2:
        break
    mean_vel += 10**(point['DBVEL']/10)
    mean_width += 10**(point['DBWIDTH']/10)
    number_of_points += 1
mean_vel /= number_of_points # log scale
mean_vel = 10*(math.log10(mean_vel))
mean_width /= number_of_points
mean_width = 10*(math.log10(mean_width))
print('Mean Velocity : ',mean_vel)
print('Mean Turbulence : ',mean_width)
print('Lowest Elevation DBZ : ',xy_points[0]['Z'])

Mean Velocity :  0.07953537095005817
Mean Turbulence :  1.668337634830106
Lowest Elevation DBZ :  0.0107378526183838
